# Chapter 30 Guided To-Do Practice Labs

**Book Reference:** Chapter 30, Section 30.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 CNN building blocks, and pass each **function or class name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch30_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import sharukh_tutor as tutor

tutor.activate(chapter="ch30", notebook="guided_todos")


---
## To-Do 30.1 (Output-Shape Arithmetic): `conv2d_output_shape(h_in, w_in, kernel_size, stride=1, padding=0)`

Implement `conv2d_output_shape(h_in, w_in, kernel_size, stride=1, padding=0)`:
- Raise `ValueError` if `stride <= 0`.
- Compute `h_out = (h_in + 2 * padding - kernel_size) // stride + 1` and `w_out = (w_in + 2 * padding - kernel_size) // stride + 1`.
- Raise `ValueError` if `h_out <= 0` or `w_out <= 0`, and otherwise return `(int(h_out), int(w_out))`.

In [ ]:
def conv2d_output_shape(h_in, w_in, kernel_size, stride=1, padding=0):
    """Compute Conv2d output (h_out, w_out) with ValueError guards."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    if stride <= 0:
        raise ValueError("stride must be positive")
    h_out = (h_in + 2 * padding - kernel_size) // stride + 1
    w_out = (w_in + 2 * padding - kernel_size) // stride + 1
    if h_out <= 0 or w_out <= 0:
        raise ValueError("output spatial dimensions must be positive")
    return (int(h_out), int(w_out))
    ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_01", conv2d_output_shape)


---
## To-Do 30.2 (From-Scratch 2D Cross-Correlation): `conv2d_single_channel_numpy(image_2d, kernel_2d, stride=1, padding=0)`

Implement pure-NumPy 2D cross-correlation matching `torch.nn.functional.conv2d` to `< 1e-5` without mutating inputs:
- Zero-pad `image_2d` on all four sides when `padding > 0`.
- Slide `kernel_2d` with step `stride` (no 180-degree kernel flip) and sum `patch * kernel_2d`.
- Return a 2D NumPy `float32` array of shape `(h_out, w_out)`.

In [ ]:
def conv2d_single_channel_numpy(image_2d, kernel_2d, stride=1, padding=0):
    """Pure-NumPy 2D cross-correlation matching PyTorch F.conv2d."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    img = np.asarray(image_2d, dtype=np.float32)
    ker = np.asarray(kernel_2d, dtype=np.float32)
    if padding > 0:
        img = np.pad(
            img, ((padding, padding), (padding, padding)), mode="constant"
        )
    kh, kw = ker.shape
    h_out = (img.shape[0] - kh) // stride + 1
    w_out = (img.shape[1] - kw) // stride + 1
    out = np.zeros((h_out, w_out), dtype=np.float32)
    for i in range(h_out):
        for j in range(w_out):
            patch = img[
                i * stride : i * stride + kh,
                j * stride : j * stride + kw,
            ]
            out[i, j] = np.sum(patch * ker)
    return out
    ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_02", conv2d_single_channel_numpy)


---
## To-Do 30.3 (Compact Medical CNN): `MedicalPatchCNN`

Implement `MedicalPatchCNN(nn.Module)` with `__init__(self, num_classes=1)`:
- Feature extractor: `Conv2d(1, 8, 3, padding=1) -> ReLU -> MaxPool2d(2) -> Conv2d(8, 16, 3, padding=1) -> ReLU -> AdaptiveAvgPool2d(1) -> Flatten`.
- Final linear head named **`self.fc`** (`nn.Linear(16, num_classes)`) so transfer-learning freezing works cleanly on `head_name="fc"`.
- `forward(self, x)` returns raw logits of shape `(B, num_classes)`.

In [ ]:
class MedicalPatchCNN(nn.Module):
    """Compact 2D CNN for single-channel medical patch classification."""

    def __init__(self, num_classes=1):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        return self.fc(self.features(x))
        ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_03", MedicalPatchCNN)


---
## To-Do 30.4 (Residual Skip Connection): `ResidualConvBlock`

Implement `ResidualConvBlock(nn.Module)` with `__init__(self, channels)`:
- Residual branch: `Conv2d(c, c, 3, padding=1) -> BatchNorm2d(c) -> ReLU -> Conv2d(c, c, 3, padding=1) -> BatchNorm2d(c)`.
- Skip connection: return `F.relu(out + x)`, preserving the exact `(B, C, H, W)` shape.

In [ ]:
class ResidualConvBlock(nn.Module):
    """Two-conv residual block with BatchNorm2d and identity skip."""

    def __init__(self, channels):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)
        ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_04", ResidualConvBlock)


---
## To-Do 30.5 (Transfer Learning): `freeze_backbone_for_transfer(model, head_name="fc")`

Implement `freeze_backbone_for_transfer(model, head_name="fc")`:
- Iterate over `model.named_parameters()` and set `p.requires_grad = name.startswith(f"{head_name}.")`.
- Count total scalar parameter elements (`p.numel()`) in each group and return `{"trainable_params": int, "frozen_params": int}`.

In [ ]:
def freeze_backbone_for_transfer(model, head_name="fc"):
    """Freeze all parameters outside `head_name.` and tally p.numel()."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    prefix = f"{head_name}."
    trainable = 0
    frozen = 0
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith(prefix)
        if param.requires_grad:
            trainable += param.numel()
        else:
            frozen += param.numel()
    return {
        "trainable_params": int(trainable),
        "frozen_params": int(frozen),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_05", freeze_backbone_for_transfer)


---
## To-Do 30.6 (Offline Medical Patch Generator): `generate_synthetic_lesion_dataset(n_samples=160, img_size=16, seed=42)`

Implement deterministic synthetic single-channel medical patch generation:
- Return `(X_tensor, y_tensor)` as `torch.float32` tensors of shapes `(n_samples, 1, img_size, img_size)` and `(n_samples,)`.
- Classes must be balanced (`0` = noisy healthy tissue background, `1` = background + bright localized circular Gaussian lesion nodule at a randomized interior `(cy, cx)` coordinate).
- Reproducible via `np.random.default_rng(seed)`.

In [ ]:
def generate_synthetic_lesion_dataset(n_samples=160, img_size=16, seed=42):
    """Generate deterministic (X[N,1,H,W], y[N]) lesion patch tensors."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    rng = np.random.default_rng(seed)
    n_neg = n_samples // 2
    n_pos = n_samples - n_neg
    labels = np.array([0] * n_neg + [1] * n_pos, dtype=np.float32)
    rng.shuffle(labels)
    yy, xx = np.meshgrid(
        np.arange(img_size), np.arange(img_size), indexing="ij"
    )
    images = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    for i, lbl in enumerate(labels):
        base = rng.uniform(0.15, 0.30)
        patch = rng.normal(base, 0.12, size=(img_size, img_size))
        if lbl == 1.0:
            cy = rng.integers(3, img_size - 3)
            cx = rng.integers(3, img_size - 3)
            amp = rng.uniform(0.95, 1.25)
            dist_sq = (yy - cy) ** 2 + (xx - cx) ** 2
            patch = patch + amp * np.exp(-dist_sq / (2.0 * 1.5 ** 2))
        images[i, 0] = np.clip(patch, 0.0, 2.0)
    return torch.from_numpy(images), torch.from_numpy(labels)
    ### END SOLUTION


In [ ]:
tutor.check("ch30_todo_06", generate_synthetic_lesion_dataset)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 30.2 (Cross-Correlation vs. Flipped Kernel):** PyTorch's `F.conv2d` slides the kernel without flipping it 180 degrees. Flipping `kernel_2d[::-1, ::-1]` breaks parity with `F.conv2d` on asymmetric filters!
- **To-Do 30.3 (`AdaptiveAvgPool2d(1)` & `self.fc`):** Hardcoding `nn.Linear(16 * 8 * 8, num_classes)` crashes the moment a hospital scanner sends a `24x24` patch instead of `16x16`. Global average pooling makes the CNN resolution-agnostic while keeping `self.fc` cleanly addressable for transfer learning.
- **To-Do 30.4 (Missing Skip Connection `+ x`):** Omitting `+ x` reduces `ResidualConvBlock` to a plain 2-layer conv block; zeroing the BatchNorm weights immediately reveals whether the identity highway passes `x` through untouched.

In [ ]:
tutor.summary()
